# Oscar — Kripto Analiz Sistemi

**3 hucre:** Kurulum → Baslat → Calistir

## Hucre 1 — Kurulum
Sadece **ilk seferde** calistir. Unsloth kurar, repo'yu ceker.

In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║  HUCRE 1 — KURULUM  (sadece ilk seferde calistir)       ║
# ╚══════════════════════════════════════════════════════════╝

# ── Unsloth guncelle ──────────────────────────────────────
import subprocess, sys
print("Unsloth guncelleniyor...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    "unsloth_zoo"
], check=False)
print("OK Unsloth guncellendi.")

# ── Config (REPO_DIR once tanimla) ───────────────────────
import os, pathlib

REPO_URL = os.environ.get(
    "PROJECT_REPO_URL",
    "https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git",
)
HF_TOKEN       = os.environ.get("HF_TOKEN", "")
REPO_DIR       = "/content/unsloth-qwen35-trading-assistant"
MODEL_NAME     = "unsloth/Qwen3-32B"
MAX_SEQ_LENGTH = 16384
DTYPE          = None
LOAD_IN_4BIT   = True

print(f"Model   : {MODEL_NAME}")
print(f"Repo    : {REPO_URL}")
print(f"HF token: {'set' if HF_TOKEN else 'not set'}")

# ── Repo clone / update ───────────────────────────────────
def _run(cmd, **kwargs):
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{output}")
    return output

def git_pull_latest():
    """Clone the repo on first run; fast-forward pull on subsequent runs."""
    repo = pathlib.Path(REPO_DIR)
    if repo.exists():
        print(_run(["git", "-C", str(repo), "pull", "--ff-only"]))
    else:
        print(_run(["git", "clone", REPO_URL, str(repo)]))
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    print(f"Repo ready at {repo}")

git_pull_latest()

# ── HF login ─────────────────────────────────────────────
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN set — public model download will still work.")

print()
print("OK Kurulum tamamlandi. Artik Hucre 2'yi calistirabilirsin.")


Unsloth guncelleniyor...
OK Unsloth guncellendi.
Model   : unsloth/Qwen3-32B
Repo    : https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git
HF token: not set
Updating 811f678..29b8853
Fast-forward
 _fix_float_parse.py  |   61 +
 _fix_save_signals.py |   84 +
 setup.ipynb          | 4403 +++++++++++++++++++++++++++++++++++++-------------
 3 files changed, 3399 insertions(+), 1149 deletions(-)
 create mode 100644 _fix_float_parse.py
 create mode 100644 _fix_save_signals.py
From https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant
   811f678..29b8853  main       -> origin/main
Repo ready at /content/unsloth-qwen35-trading-assistant
No HF_TOKEN set — public model download will still work.

OK Kurulum tamamlandi. Artik Hucre 2'yi calistirabilirsin.


## Hucre 2 — Baslat
**Her session'da** calistir. Modeli yukler, fonksiyonlari hazirlar. ~10-15 dk.

In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║  HUCRE 2 — BASLAT  (her session'da calistir)            ║
# ╚══════════════════════════════════════════════════════════╝

# ── Repo guncelle (her session basinda) ──────────────────────────────────
import subprocess
_pull = subprocess.run(
    ["git", "-C", "/content/unsloth-qwen35-trading-assistant", "pull", "--ff-only"],
    capture_output=True, text=True
)
print("Repo:", (_pull.stdout + _pull.stderr).strip())

import os, sys, re, json, time as _time, threading as _threading
import requests as _requests
from pathlib import Path
from datetime import datetime, timezone

# ── Config ────────────────────────────────────────────────
import os

REPO_URL = os.environ.get(
    "PROJECT_REPO_URL",
    "https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git",
)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Model seçimi ────────────────────────────────────────────────────────────
# Qwen3-32B: Qwen3.5-27B'ye göre daha güçlü reasoning, native thinking modu,
#            A100 80GB'de 4-bit ile ~18GB VRAM kullanır.
MODEL_NAME     = "unsloth/Qwen3-32B"
MAX_SEQ_LENGTH = 16384   # 16k context — reasoning zinciri için gerekli
DTYPE          = None    # auto-detect (BF16 on A100)
LOAD_IN_4BIT   = True    # ~18GB VRAM

REPO_DIR = "/content/unsloth-qwen35-trading-assistant"

print(f"Model   : {MODEL_NAME}")
print(f"Context : {MAX_SEQ_LENGTH} tokens")
print(f"4-bit   : {LOAD_IN_4BIT}")
print(f"Repo    : {REPO_URL}")
print(f"HF token: {'set' if HF_TOKEN else 'not set'}")

# ── Repo guncelle ─────────────────────────────────────────
import subprocess, pathlib, sys

def _run(cmd: list[str], **kwargs) -> str:
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{output}")
    return output

def git_pull_latest():
    """Clone the repo on first run; fast-forward pull on subsequent runs."""
    repo = pathlib.Path(REPO_DIR)
    if repo.exists():
        print(_run(["git", "-C", str(repo), "pull", "--ff-only"]))
    else:
        print(_run(["git", "clone", REPO_URL, str(repo)]))
    # Add repo to Python path so `src` is importable
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    print(f"Repo ready at {repo}")

git_pull_latest()

# ── Model yukle ───────────────────────────────────────────
# ── Model: Drive cache + HF mirror (ilk seferinde indirir, sonra Drive'dan yukler) ───
import subprocess, os
# hf-mirror.com = HuggingFace CDN mirror, unsloth modelleri icin gerekli
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# Drive cache — bir kez indirir, sonra hep Drive'dan yukler
DRIVE_CACHE = "/content/drive/MyDrive/oscar_model_cache"
os.makedirs(DRIVE_CACHE, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_CACHE

# ModelScope'u devre disi birak (unsloth/ modelleri orada yok)
os.environ.pop("UNSLOTH_USE_MODELSCOPE", None)

import glob as _glob
_cached = _glob.glob(f"{DRIVE_CACHE}/**/*Qwen3*72B*", recursive=True)
if _cached:
    print(f"Drive cache bulundu — Drive'dan yukleniyor (~2-3 dk)")
else:
    print("Ilk yuklem — hf-mirror'dan Drive'a indiriliyor (~15-20 dk, bir kez)")

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

FastLanguageModel.for_inference(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

# ── Chat fonksiyonu ───────────────────────────────────────
import sys, re
sys.path.insert(0, REPO_DIR)

from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages

_conversation_history = []
_text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def _strip_thinking(text: str) -> str:
    """
    Qwen3 <think>...</think> bloğunu temizle, sadece asıl cevabı döndür.
    Thinking içeriği tamamen kaldırılır — kullanıcıya sade, temiz çıktı gider.
    """
    # <think>...</think> bloğunu (multiline) kaldır
    cleaned = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.DOTALL)
    return cleaned.strip()


def chat(
    user_message: str,
    market_context: str = "",
    reset: bool = False,
    thinking: bool = True,
    max_new_tokens: int = 2048,
    temperature: float = 0.6,
    top_p: float = 0.95,
    system_override: str | None = None,
):
    """
    Atlas Trade ile sohbet et.

    Parametreler
    ------------
    user_message    : Sorun veya analiz isteğin.
    market_context  : fetch_context() veya fetch_multi_tf_context() çıktısı.
    reset           : True → geçmiş sıfırlanır, yeni konuya başlanır.
    thinking        : True → model önce sessiz muhakeme yapar (önerilen).
    max_new_tokens  : Maksimum üretilecek token (thinking dahil).
    """
    global _conversation_history
    if reset:
        _conversation_history = []
        print("[sıfırlandı]")

    messages = build_trading_messages(
        user_message=user_message,
        market_context=market_context,
        history=_conversation_history,
        system_override=system_override,
    )

    # Chat template — enable_thinking Qwen3 serisinde desteklenir
    try:
        text = _text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=thinking,
        )
    except TypeError:
        # Eski transformers sürümü enable_thinking bilmiyorsa sessizce geç
        text = _text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = _text_tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids      = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        pad_token_id   = _text_tokenizer.eos_token_id,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_p          = top_p,
        do_sample      = True,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw        = _text_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    response   = _strip_thinking(raw)

    _conversation_history.append({"role": "user",      "content": user_message})
    _conversation_history.append({"role": "assistant", "content": response})
    return response


def clear_history():
    global _conversation_history
    _conversation_history = []
    print("[geçmiş temizlendi]")


print("✅ chat() hazır")
print("   thinking modu: AÇIK (varsayılan)")
print("   max_new_tokens: 2048")

# ── Drive baglantisi ──────────────────────────────────────
from google.colab import drive
import json, os
from pathlib import Path
from datetime import datetime, timezone

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

OSCAR_DUMAN_DIR = Path("/content/drive/MyDrive/oscar_duman")
OSCAR_DUMAN_DIR.mkdir(exist_ok=True)

def _load(fname):
    p = OSCAR_DUMAN_DIR / fname
    if not p.exists():
        return [] if fname == "trade_history.json" or fname == "reflections.json" else {}
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def _save(fname, data):
    p = OSCAR_DUMAN_DIR / fname
    tmp = p.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    tmp.replace(p)

def load_market_data():   return _load("market_data.json")
def load_active_signals(): return _load("active_signals.json")
def load_trade_history():
    d = _load("trade_history.json")
    return d if isinstance(d, list) else []
def save_signal(signal):
    signals = load_active_signals()
    signals[signal["sinyal_id"]] = signal
    _save("active_signals.json", signals)
def load_reflections():
    d = _load("reflections.json")
    return d if isinstance(d, list) else []
def save_reflection(ref):
    refs = load_trade_history()
    refs2 = _load("reflections.json")
    if not isinstance(refs2, list): refs2 = []
    refs2.append(ref)
    _save("reflections.json", refs2)

print(f"Drive baglandi: {OSCAR_DUMAN_DIR}")
md = load_market_data()
guncelleme = md.get("_guncelleme", "yok")
print(f"Piyasa verisi : {len(md)} coin  |  Guncelleme: {guncelleme}")
print(f"Aktif sinyal  : {len(load_active_signals())}")
print(f"Islem gecmisi : {len(load_trade_history())}")


# ── Trigger (Oscar karar kuyruğu) ─────────────────────────────────────────
TRIGGERS_FILE = OSCAR_DUMAN_DIR / "triggers.json"

def load_triggers() -> dict:
    if not TRIGGERS_FILE.exists():
        return {}
    try:
        return json.loads(TRIGGERS_FILE.read_text(encoding="utf-8"))
    except Exception:
        return {}

def set_trigger_decision(sinyal_id: str, karar: str) -> None:
    """Oscar kararini triggers.json'a yaz: GIR / PAS / IPTAL."""
    triggers = load_triggers()
    if sinyal_id in triggers:
        triggers[sinyal_id]["karar"]        = karar
        triggers[sinyal_id]["karar_zamani"] = datetime.utcnow().isoformat()
        TRIGGERS_FILE.write_text(
            json.dumps(triggers, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        print(f"[{sinyal_id}] Karar yazildi: {karar}")
    else:
        print(f"[{sinyal_id}] Trigger bulunamadi.")

print("OK Trigger yardimcilari hazir.")


# ── Sinyal uretici ────────────────────────────────────────
import re
from datetime import datetime, timezone

def _utcnow():
    return datetime.now(timezone.utc).isoformat()

def _to_float(s):
    s = str(s).strip().replace(",", "").replace("$", "").replace(" ", "")
    if s.lower().endswith("k"): return float(s[:-1]) * 1000
    if s.lower().endswith("m"): return float(s[:-1]) * 1_000_000
    return float(s)

def _parse_kosul_block(analysis: str) -> dict:
    """```KOSUL ... ``` blogunu parse et."""
    m = re.search(r"```KOSUL[ \t]*\n(.+?)```", analysis, re.DOTALL | re.IGNORECASE)
    if not m:
        return {}
    block = m.group(1)
    result = {}
    for line in block.strip().splitlines():
        if ":" in line:
            key, _, val = line.partition(":")
            result[key.strip().upper()] = val.strip()
    return result

def _parse_signal(analysis: str, symbol: str) -> dict | None:
    """Oscar analiz metninden sinyal cikar — once KOSUL blogu, sonra regex."""
    NUM = r"[$]?[\d][\d,.]*[kKmM]?"

    kb = _parse_kosul_block(analysis)

    def kb_float(key):
        try: return _to_float(kb[key]) if key in kb else None
        except: return None

    yon         = kb.get("YON", "").upper() or None
    entry       = kb_float("ENTRY")
    stop        = kb_float("STOP")
    tp1         = kb_float("HEDEF_1")
    tp2         = kb_float("HEDEF_2")
    kosul_tipi  = kb.get("TIP", "DIRECT").upper()
    kosul_fiyat = kb_float("TETIK") or entry

    if not yon:
        text_lower  = analysis.lower()
        long_score  = text_lower.count("long") + text_lower.count("bullish")
        short_score = text_lower.count("short") + text_lower.count("bearish")
        if   long_score  > short_score: yon = "LONG"
        elif short_score > long_score:  yon = "SHORT"

    def find(patterns):
        for pat in patterns:
            m = re.search(pat, analysis, re.IGNORECASE)
            if m:
                try: return _to_float(m.group(1))
                except: pass
        return None

    if not entry:
        entry = find([r"[Ee]ntry[\s*:]+(" + NUM + ")", r"[Gg]iri[sş][\s*:]+(" + NUM + ")"])
    if not stop:
        stop  = find([r"[Ss]top[\s*:]+(" + NUM + ")", r"\bSL[\s*:]+(" + NUM + ")"])
    if not tp1:
        tp1   = find([r"[Hh]edef\s*1[\s*:]+(" + NUM + ")", r"TP\s*1[\s*:]+(" + NUM + ")"])
    if not tp2:
        tp2   = find([r"[Hh]edef\s*2[\s*:]+(" + NUM + ")", r"TP\s*2[\s*:]+(" + NUM + ")"])

    if not all([yon, entry, stop, tp1]):
        return None

    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    return {
        "sinyal_id":        f"{symbol}_{ts}",
        "symbol":           symbol,
        "yon":              yon,
        "entry":            entry,
        "stop":             stop,
        "hedef_1":          tp1,
        "hedef_2":          tp2 or tp1,
        "kosul_tipi":       kosul_tipi,
        "kosul_fiyat":      kosul_fiyat,
        "durum":            "bekliyor",
        "analiz":           analysis,
        "olusturma_zamani": _utcnow(),
    }


def _build_context_from_cache(symbol: str) -> str:
    md = load_market_data()
    if symbol not in md:
        return ""
    d = md[symbol]
    lines = [f"Sembol: {symbol}"]
    for tf, tfd in d.get("timeframes", {}).items():
        lines.append(f"\n── {tf.upper()} ─────────────")
        lines.append(f"Fiyat   : {tfd.get('price', 0):.2f}")
        lines.append(f"RSI14   : {tfd.get('rsi14', 0):.1f}")
        lines.append(f"ATR14   : {tfd.get('atr14', 0):.4f}")
        lines.append(f"EMA bias: {tfd.get('ema_bias', '?')}")
        lines.append(f"MACD hist: {tfd.get('macd_hist', 0):.4f}")
    try:
        funding_pct = float(d.get("funding_rate", 0)) * 100
        lines.append(f"\nFunding : {funding_pct:.4f}%")
    except Exception:
        pass
    lines.append(f"OI      : {d.get('open_interest', 'N/A')}")
    lines.append(f"Veri ts : {d.get('ts', 'N/A')}")
    liq = d.get("liquidation", {})
    if liq:
        lines.append("")
        lines.append("── LIKIDASYON ──────────────────────")
        cascade = liq.get("cascade_riski", "")
        if cascade in ("YUKSEK", "YÜKSEK"):
            lines.append("UYARI: CASCADE RISKI YUKSEK")
        long_z = liq.get("long_liq_zones", [])
        short_z = liq.get("short_liq_zones", [])
        if long_z:
            z = long_z[0]
            lines.append(f"Long liq havuzu : ${z['price']:,.1f}  (${z['hacim_usd']:,}  — %{z['uzaklik_pct']} uzakta)")
        if short_z:
            z = short_z[0]
            lines.append(f"Short liq havuzu: ${z['price']:,.1f}  (${z['hacim_usd']:,}  — %{z['uzaklik_pct']} uzakta)")
        oi_deg = liq.get("oi_degisim_pct", 0)
        lines.append(f"OI 1s degisim   : %{oi_deg:+.2f}  |  Stres: {liq.get('stres_skoru', 0)}/10")
        bar = liq.get("bid_ask_ratio", 1.0)
        lines.append(f"Alis/Satis      : {bar}")
    return "\n".join(lines)


def oscar_signal(symbol: str, timeframe: str = "4h", soru: str = "") -> dict | None:
    if not soru:
        soru = (
            "Bu sembol icin net bir trade sinyali ver. "
            "Entry, stop ve iki hedef belirt. "
            "KOSUL blogunu mutlaka doldur."
        )

    ctx = _build_context_from_cache(symbol)
    if not ctx:
        from src.market_data import fetch_multi_tf_context
        print(f"[{symbol}] Cache yok — canli veri cekiliyor...")
        ctx = fetch_multi_tf_context(symbol)

    print(f"[{symbol}] Oscar analiz yapiyor...")
    analiz = chat(soru, market_context=ctx, reset=True, thinking=False)
    print("\n" + analiz)
    print("─" * 55)

    sinyal = _parse_signal(analiz, symbol)
    if sinyal:
        save_signal(sinyal)
        print(f"\n OK Sinyal Drive'a yazildi: {sinyal['sinyal_id']}")
        print(f"   {sinyal['yon']}  Entry:{sinyal['entry']}  Stop:{sinyal['stop']}")
        print(f"   TP1:{sinyal['hedef_1']}  TP2:{sinyal['hedef_2']}")
        print(f"   Kosul: {sinyal['kosul_tipi']} @ {sinyal['kosul_fiyat']}")
    else:
        print("\n UYARI Sinyal parse edilemedi.")
    return sinyal


print("OK oscar_signal() v2 hazir — kosul destekli")


# ── Yansima motoru ────────────────────────────────────────
def oscar_reflect(trade: dict) -> dict:
    """Kapanan islem icin zengin oz-degerlendirme — fine-tuning ham maddesi."""
    sid   = trade.get("sinyal_id", "?")
    sym   = trade.get("symbol", "?")
    yon   = trade.get("yon", "?")
    entry = trade.get("entry", "?")
    stop  = trade.get("stop", "?")
    tp1   = trade.get("hedef_1", "?")
    tp2   = trade.get("hedef_2", "?")
    sonuc = trade.get("sonuc", "?")
    r     = trade.get("r_ratio", "?")
    pnl   = trade.get("pnl_usd", "?")
    kosul_tipi  = trade.get("kosul_tipi", "DIRECT")
    kosul_fiyat = trade.get("kosul_fiyat", "?")

    # Guncel piyasa verisini al (kiyaslama icin)
    ctx = _build_context_from_cache(sym)

    prompt = f"""Asagidaki islemi derinlemesine analiz et. Bu analiz seni egitmek icin kullanilacak — donuk ve yuzeysel olma.

═══ ISLEM DETAYLARI ═══
Sembol     : {sym}
Yon        : {yon}
Entry      : {entry}
Stop       : {stop}
Hedef 1    : {tp1}
Hedef 2    : {tp2}
Sonuc      : {sonuc}
R Orani    : {r}
P&L        : {pnl} USD
Kosul Tipi : {kosul_tipi} @ {kosul_fiyat}

═══ KURULAN ANALIZ ═══
{trade.get("analiz", "")[:600]}

═══ GUNCEL PIYASA ═══
{ctx}

Simdi su sorulari tek tek yanit ver:

1. KOSUL ANALIZI: Koydugun kosul ({kosul_tipi} @ {kosul_fiyat}) mantikli miydi? Neden bu kosulu sectim?
2. SONUC DEGERLENDIRME: {sonuc} cikti — bu beklenen mi yoksa surpriz miydi? Neden?
3. YANLIS OKUMA: Hangi indikatoru veya sinyali yanlis yorumladim? Spesifik ol.
4. PIYASA YAPISI: O an piyasada ne oluyordu? Gozden kactirdim mi?
5. OGRENME: Bir dahaki benzer setupta ne farkli yapacagim? 1 somut kural yaz.
6. MODEL HUKMU: Bu islem iyi bir setup muydu (kalite 1-10)? Neden?"""

    response = chat(prompt, thinking=True, reset=True)

    reflection = {
        "sinyal_id":  sid,
        "symbol":     sym,
        "yon":        yon,
        "sonuc":      sonuc,
        "r_ratio":    r,
        "pnl_usd":    pnl,
        "kosul_tipi": kosul_tipi,
        "yansiima":   response,
        "ts":         datetime.utcnow().isoformat(),
    }
    save_reflection(reflection)
    print(f"[{sid}] Yansiima kaydedildi. Sonuc: {sonuc} | R: {r} | P&L: {pnl}")
    return reflection


def reflect_unprocessed(limit: int = 5) -> int:
    """Yansiitilmamis kapali islemleri isle."""
    history     = load_trade_history()
    reflections = load_reflections()
    reflected   = {r["sinyal_id"] for r in reflections}

    pending = [t for t in history if t.get("sinyal_id") not in reflected]
    if not pending:
        print("Yansiitilacak yeni islem yok.")
        return 0

    print(f"{len(pending)} islem yansiitilacak (limit={limit})...")
    done = 0
    for trade in pending[:limit]:
        try:
            oscar_reflect(trade)
            done += 1
        except Exception as e:
            print(f"Hata [{trade.get('sinyal_id')}]: {e}")
    return done


reflect_unprocessed(limit=3)


print()
print("=" * 55)
print("  Oscar hazir. Hucre 3'u calistir.")
print("=" * 55)


# ── Multi-Agent Oscar ─────────────────────────────────────────────────────────
from src.trading_prompt import (
    TECH_AGENT_PROMPT, DERIV_AGENT_PROMPT,
    LIQ_AGENT_PROMPT, META_AGENT_PROMPT, PREFILTER_PROMPT
)

def _extract_score(text: str) -> int:
    """SKOR: X satirindan skoru cek."""
    m = re.search(r"SKOR\s*:\s*(\d+)", text)
    return int(m.group(1)) if m else 0

def _extract_direction(text: str) -> str:
    m = re.search(r"YON\s*:\s*(LONG|SHORT|TARAFSIZ)", text)
    return m.group(1) if m else "TARAFSIZ"

def oscar_prefilter(symbol: str, ctx: str) -> tuple[int, str]:
    """Hizli on eleme — 5 saniye, thinking=False."""
    soru = f"{symbol} icin hizli tarama yap."
    result = chat(soru, market_context=ctx, reset=True,
                  thinking=False, system_override=PREFILTER_PROMPT)
    score = _extract_score(result)
    direction = _extract_direction(result)
    return score, direction

def oscar_signal_v2(symbol: str) -> dict | None:
    """
    Multi-agent sinyal uretici.
    1. Pre-filter (hizli) — skor < 6 ise atla
    2. Teknik Ajan (ICT/SMC)
    3. Turev Ajani (Funding/OI/CVD)
    4. Likit Ajani (Cascade/Sweep)
    5. Meta-Oscar (sentez + KOSUL)
    """
    import sys
    sys.path.insert(0, REPO_DIR)
    from src.market_data import fetch_multi_tf_context
    from src.liquidation import get_liquidation_context, format_for_oscar

    # Piyasa verisi cek
    try:
        ctx = fetch_multi_tf_context(symbol)
    except Exception as e:
        print(f"[{symbol}] Veri hatasi: {e}")
        return None

    # Likidasyon verisi
    try:
        price_match = re.search(r"Son fiyat[^:]*:\s*([\d.]+)", ctx)
        price = float(price_match.group(1)) if price_match else 0.0
        liq = get_liquidation_context(symbol, price, funding=0.0)
        liq_text = format_for_oscar(liq, symbol)
    except Exception:
        liq_text = ""

    # 1. Pre-filter
    score, direction = oscar_prefilter(symbol, ctx)
    print(f"  [{symbol}] Pre-filter: {score}/10 — {direction}")
    if score < 6:
        return None

    ctx_with_liq = ctx + ("\n\n" + liq_text if liq_text else "")

    # 2. Teknik Ajan
    tech = chat(
        f"{symbol} teknik analiz yap.",
        market_context=ctx, reset=True,
        thinking=False, system_override=TECH_AGENT_PROMPT
    )

    # 3. Turev Ajani
    deriv = chat(
        f"{symbol} turev piyasasi analizi yap.",
        market_context=ctx, reset=True,
        thinking=False, system_override=DERIV_AGENT_PROMPT
    )

    # 4. Likidasyon Ajani
    liq_analysis = chat(
        f"{symbol} likidasyon cascade analizi yap.",
        market_context=ctx_with_liq, reset=True,
        thinking=False, system_override=LIQ_AGENT_PROMPT
    )

    # 5. Meta-Oscar sentez (thinking=True)
    sentez = (
        f"SEMBOL: {symbol}\n\n"
        f"=== TEKNİK AJAN ===\n{tech}\n\n"
        f"=== TÜREV AJANI ===\n{deriv}\n\n"
        f"=== LİKİDASYON AJANI ===\n{liq_analysis}"
    )
    final = chat(
        f"{symbol} icin 3 ajan raporunu sentezle ve nihai karar ver.",
        market_context=sentez, reset=True,
        thinking=True, system_override=META_AGENT_PROMPT
    )

    print(f"  [{symbol}] Meta-Oscar cevabi alindi")

    # KOSUL blogunu parse et
    m = re.search(r"```KOSUL[ \t]*\n(.+?)```", final, re.DOTALL)
    if not m:
        print(f"  [{symbol}] KOSUL blogu bulunamadi")
        return None

    lines = m.group(1).strip().splitlines()
    kosul = {}
    for line in lines:
        if ":" in line:
            k, _, v = line.partition(":")
            kosul[k.strip()] = v.strip()

    if kosul.get("TIP") in (None, "NONE", ""):
        print(f"  [{symbol}] KAÇIN/BEKLE — sinyal yok")
        return None

    def _f(val: str) -> float:
        """Virgul ve bosluklu sayilari float'a cevirir: '74,502.3' -> 74502.3"""
        try:
            return float(str(val).replace(",", "").replace(" ", "").strip() or "0")
        except ValueError:
            return 0.0

    try:
        return {
            "symbol":      symbol,
            "yon":         kosul.get("YON", ""),
            "kosul_tipi":  kosul.get("TIP", "DIRECT"),
            "kosul_fiyat": _f(kosul.get("TETIK", "0")),
            "entry":       _f(kosul.get("ENTRY", "0")),
            "stop":        _f(kosul.get("STOP", "0")),
            "hedef_1":     _f(kosul.get("HEDEF_1", "0")),
            "hedef_2":     _f(kosul.get("HEDEF_2", "0")),
            "analiz":      final,
            "prefilter_score": score,
        }
    except Exception as e:
        print(f"  [{symbol}] Parse hatasi: {e}")
        return None

print("Multi-agent Oscar hazir.")


Repo: Already up to date.
Model   : unsloth/Qwen3-32B
Context : 16384 tokens
4-bit   : True
Repo    : https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git
HF token: not set
Already up to date.
Repo ready at /content/unsloth-qwen35-trading-assistant
Drive cache bulundu — Drive'dan yukleniyor (~2-3 dk)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - i

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

unsloth/qwen3-32b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: unsloth/Qwen3-32B
Max sequence length: 16384
✅ chat() hazır
   thinking modu: AÇIK (varsayılan)
   max_new_tokens: 2048
Drive baglandi: /content/drive/MyDrive/oscar_duman
Piyasa verisi : 0 coin  |  Guncelleme: yok
Aktif sinyal  : 86
Islem gecmisi : 0
OK Trigger yardimcilari hazir.
OK oscar_signal() v2 hazir — kosul destekli
Yansiitilacak yeni islem yok.

  Oscar hazir. Hucre 3'u calistir.
Multi-agent Oscar hazir.


## Hucre 3 — Calistir
Hucre 2 bittikten sonra calistir. Oscar taramaya baslar.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  HUCRE 3 — CALISTIR  (Hucre 2'den sonra)                ║
# ║  Durdurmak icin: Interrupt (kare buton)                 ║
# ╚══════════════════════════════════════════════════════════╝

# ── Karar motoru + Tarama dongusu ────────────────────────
import time as _time
import threading as _threading

def oscar_decide_trigger(trigger: dict) -> str:
    """Tetiklenmis sinyal icin Oscar GIR/PAS/IPTAL kararı verir."""
    sid         = trigger["sinyal_id"]
    sym         = trigger.get("symbol", "")
    yon         = trigger.get("yon", "")
    analiz_ozet = trigger.get("analiz", "")[:400]
    ctx         = _build_context_from_cache(sym)

    prompt = f"""Daha once {sym} icin bir sinyal kurmuştun.
Simdi giris kosulu olustu. Guncel piyasaya bak ve karar ver.

Orijinal analizin (ozet):
{analiz_ozet}

Sinyal detaylari:
Yon    : {yon}
Entry  : {trigger.get("entry")}
Stop   : {trigger.get("stop")}
Hedef1 : {trigger.get("hedef_1")}
Hedef2 : {trigger.get("hedef_2")}
Kosul  : {trigger.get("kosul_tipi")} @ {trigger.get("kosul_fiyat")}

Guncel piyasa:
{ctx}

Karar ver — sadece su uctan birini yaz ve 2 cumleyle acikla:
GIR   — kosullar hala gecerli, isleme gir
PAS   — kosul olustu ama setup bozuldu, bu sinyali atla
IPTAL — analiz tamamen gecersiz, sinyali sil"""

    karar_text = chat(prompt, thinking=True, reset=True)
    print(f"[{sid}] Oscar karar metni:\n{karar_text[:300]}")

    ku = karar_text.upper()
    if "GIR" in ku or "GİR" in ku:
        return "GİR"
    elif "IPTAL" in ku or "İPTAL" in ku:
        return "İPTAL"
    else:
        return "PAS"


def _decision_loop(check_interval: int = 30):
    """Arka plan dongusu — tetikleyici gelince Oscar karar verir."""
    print("Oscar karar motoru baslatildi.")
    while True:
        try:
            triggers = load_triggers()
            pending  = {sid: t for sid, t in triggers.items()
                        if t.get("karar") is None}
            if pending:
                print(f"{len(pending)} bekleyen karar...")
                for sid, trigger in pending.items():
                    try:
                        karar = oscar_decide_trigger(trigger)
                        set_trigger_decision(sid, karar)
                        print(f"[{sid}] Karar: {karar}")
                        emoji = "GIR" if "GIR" in karar or "GIR" in karar else ("PAS" if "PAS" in karar else "IPTAL")
                        sym = trigger.get("symbol", "?")
                        _tg(f"{emoji}: Oscar Karari {karar} — {sym} {trigger.get('yon','?')} @ {trigger.get('entry','?')}")
                    except Exception as e:
                        print(f"[{sid}] Karar hatasi: {e}")
                        _tg(f"UYARI Karar hatasi [{sid}]: {e}")
        except Exception as e:
            print(f"Karar dongusu hatasi: {e}")
        _time.sleep(check_interval)


_threading.Thread(target=_decision_loop, daemon=True).start()
print("OK Oscar karar motoru arka planda calisiyor.")
print("   Siradaki adim: run_auto_loop() ile taramayı baslat.")



# ── Telegram bildirimi (Oscar) ────────────────────────────────────────────
import requests as _requests

def _tg(text: str) -> None:
    """Oscar Telegram bildirimi. Token/chat yoksa sessizce atla."""
    token = "8680964677:AAGp9MHbMKsAZRYH-Z_3yt9wQELCe6MLQYo"
    chat  = "524569814"
    try:
        _requests.post(
            f"https://api.telegram.org/bot{token}/sendMessage",
            json={"chat_id": chat, "text": text, "parse_mode": "HTML"},
            timeout=8,
        )
    except Exception:
        pass

import time as _time

SCAN_INTERVAL_MIN = 15   # dakika: kac dakikada bir tum coinler taransin
COINS_TO_SCAN = [
    # Buyuk cap
    "BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
    # Orta cap
    "AVAXUSDT", "DOGEUSDT", "ADAUSDT", "LINKUSDT", "DOTUSDT",
    # Yuksek volatilite
    "INJUSDT", "SUIUSDT", "NEARUSDT", "ATOMUSDT", "LTCUSDT",
]

def _scan_all() -> list[dict]:
    """Tum coinleri tara, sinyal uret, Drive'a kaydet."""
    import uuid
    signals = []
    for sym in COINS_TO_SCAN:
        try:
            sig = oscar_signal_v2(sym)
            if sig:
                # Drive'a kaydet
                sig["sinyal_id"] = f"{sym}_{uuid.uuid4().hex[:8]}"
                sig["durum"]     = "bekliyor"
                try:
                    add_signal(sig)
                except Exception as de:
                    print(f"  [{sym}] Drive kayit hatasi: {de}")

                signals.append(sig)
                yon   = sig.get("yon", "?")
                entry = sig.get("entry", "?")
                kosul = sig.get("kosul_tipi", "DIRECT")
                tetik = sig.get("kosul_fiyat", "?")
                _tg(f"📡 <b>Sinyal: {sym}</b>\nYon: {yon} | Entry: {entry}\nKosul: {kosul} @ {tetik}")
            else:
                _tg(f"⏩ {sym} — sinyal yok / pas")
            _time.sleep(2)
        except Exception as e:
            print(f"  [{sym}] Hata: {e}")
            _tg(f"⚠️ {sym} analiz hatasi: {e}")
    return signals


def run_auto_loop():
    """Tarama dongusu — Ctrl+C ile durdur."""
    print("Otomatik tarama basladi.")
    print(f"Interval: {SCAN_INTERVAL_MIN} dakika | Coins: {len(COINS_TO_SCAN)}")
    cycle = 0
    while True:
        cycle += 1
        ts = datetime.utcnow().strftime("%H:%M UTC")
        print(f"\n--- Tarama #{cycle}  {ts} ---")
        _tg(f"🔍 <b>Tarama #{cycle}</b> basliyor — {ts}\n{len(COINS_TO_SCAN)} coin")
        try:
            signals = _scan_all()
            print(f"Bu turda {len(signals)} sinyal uretildi.")
            _tg(f"✅ Tarama #{cycle} tamamlandi — {len(signals)} sinyal uretildi")
        except Exception as e:
            print(f"Tarama hatasi: {e}")
            _tg(f"❌ Tarama #{cycle} hatasi: {e}")

        # Yansiimalari isle (en fazla 5 adet)
        try:
            reflect_unprocessed(limit=5)
        except Exception as e:
            print(f"Yansiima hatasi: {e}")

        # Aktif sinyal ozeti
        aktif = load_active_signals()
        print(f"Aktif sinyal sayisi: {len(aktif)}")

        print(f"Sonraki tarama {SCAN_INTERVAL_MIN} dakika sonra...")
        _time.sleep(SCAN_INTERVAL_MIN * 60)


# Donguyu baslat (hucreyi kesmek icin Colab interrupt kullan)
run_auto_loop()


Oscar karar motoru baslatildi.
OK Oscar karar motoru arka planda calisiyor.
   Siradaki adim: run_auto_loop() ile taramayı baslat.
Otomatik tarama basladi.
Interval: 15 dakika | Coins: 15

--- Tarama #1  18:27 UTC ---


/tmp/ipykernel_112083/4109987234.py:150: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M UTC")
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[sıfırlandı]


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

  [BTCUSDT] Pre-filter: 0/10 — TARAFSIZ


...

[sıfırlandı]


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [BNBUSDT] Pre-filter: 6/10 — SHORT
[sıfırlandı]


Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[sıfırlandı]
